In [0]:
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split
from config import VALIDATION_SIZE, get_run_id

In [0]:
run_id = get_run_id()
print(run_id)

In [0]:
# Read the table from Unity Catalog
spark_df = spark.table("CompVis_OD.data.image_paths")

dataset_source = mlflow.data.from_spark(
    spark_df,
    table_name = "CompVis_OD.data.image_paths",
    targets = "label"
)

df = spark_df.toPandas()
full_paths = df["image_path"].tolist()
full_labels = df["label"].tolist()

In [0]:
# Split the data into train and validation sets using stratification based on the labels to make sure the distribution of labels is similar in both sets

X_train_paths, X_val_paths, y_train, y_val = train_test_split(
    full_paths,
    full_labels,
    test_size = VALIDATION_SIZE,
    stratify = full_labels,
    random_state = 14
)

In [0]:
with mlflow.start_run(run_id = run_id):
    mlflow.log_input(dataset_source, context = 'training')
    mlflow.log_params({
        "validation_size": VALIDATION_SIZE,
        "random_state": 14,
    })
    mlflow.log_metrics({
        "training_samples_count": len(X_train_paths),
        "validation_samples_count": len(X_val_paths)
    })

In [0]:
# Combine split metadata
train_df = pd.DataFrame({"path": X_train_paths, "label": y_train, "split": "train"})
val_df = pd.DataFrame({"path": X_val_paths, "label": y_val, "split": "val"})
splits_df = pd.concat([train_df, val_df], ignore_index=True)

# Save to Unity Catalog Volume
splits_path = "/Volumes/compvis_od/data/ovarian_dataset/processed/splits.parquet"
splits_df.to_parquet(splits_path, index=False)